In [ ]:
import pandas as pd
from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema
from pyiceberg.types import StringType
from pyiceberg.schema import NestedField

catalog = load_catalog(
    "bronze",
    **{
        "type": "sql",
        "uri": "sqlite:////data/data_files/iceberg/iceberg_warehouse/pyiceberg_catalog.db",
        "warehouse": "file:///data/data_files/iceberg/iceberg_warehouse",
    },
)

schema = Schema(
    NestedField(field_id=1, name="ContactType", type=StringType(), required=False),
    NestedField(field_id=2, name="PersonID", type=StringType(), required=False),
    NestedField(field_id=3, name="ModifiedDate", type=StringType(), required=False)
)
print(catalog.name)

catalog.create_table("person.ContactType", schema=schema)

table = catalog.load_table("person.ContactType")
df = pd.read_iceberg(table, catalog_name="bronze")
print(df.head())

In [ ]:
import pandas as pd
from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema
from pyiceberg.types import StringType
from pyiceberg.schema import NestedField
from pyiceberg.catalog.sql import SqlCatalog

catalog: SqlCatalog = load_catalog(
    "bronze",
    **{
        "type": "sql",
        "uri": "sqlite:////data/data_files/iceberg/iceberg_warehouse/pyiceberg_catalog.db",
        "warehouse": "file:///data/data_files/iceberg/iceberg_warehouse",
    },
)

schema = Schema(
    NestedField(field_id=1, name="ContactType", type=StringType(), required=False),
    NestedField(field_id=2, name="PersonID", type=StringType(), required=False),
    NestedField(field_id=3, name="ModifiedDate", type=StringType(), required=False)
)

print(f"Catalog name: {catalog.name}")

# --- FIX: Create the namespace first ---
namespace_identifier = ('person',)
if namespace_identifier not in catalog.list_namespaces():
    print(f"Creating namespace: {namespace_identifier}")
    catalog.create_namespace(namespace_identifier)
# -------------------------------------

# Now you can create the table within the existing namespace
catalog.create_table("person.ContactType", schema=schema)
print("Table created successfully.")

table = catalog.load_table("person.ContactType")

# Note: The table is empty right now, so head() will likely return an empty DataFrame
df = pd.read_iceberg(table) 
# You can also pass the catalog object directly to pandas.read_iceberg() if needed
# df = pd.read_iceberg(table_identifier="person.ContactType", catalog=catalog)

print(df.head())

In [ ]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog(
    "bronze",
    **{
        "type": "hive",
        "uri": "thrift://localhost:9083",  # Replace with your HMS host:port
        "warehouse": "file:///data/data_files/iceberg/iceberg_warehouse",
    },
)


table = catalog.load_table("manual_Person.ContactType")